<a href="https://colab.research.google.com/github/saadhana192465019/Digital-Forensics-and-cyber-crime-Investigation--CSA6102/blob/main/Experiment_14_Phishing_Email_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import re
from urllib.parse import urlparse
import matplotlib.pyplot as plt

In [2]:
from google.colab import files

uploaded = files.upload()

Saving phishing_emails.csv to phishing_emails.csv


In [4]:
df = pd.read_csv("phishing_emails.csv")
df.head()

,Email_ID,From,Reply_To,Return_Path,Subject,Body,Label
0,1,support@paypa1.com,help@gmail.com,bounce@paypa1.com,Verify Your PayPal Account,"Dear customer, your PayPal account has been li...",Phishing
1,2,security@amazon.com,security@amazon.com,bounce@amazon.com,Amazon Order Confirmation,Your Amazon order has been successfully placed...,Legitimate
2,3,admin@micr0soft-support.com,support@yahoo.com,admin@micr0soft-support.com,Microsoft Security Alert,Unusual login detected. Verify your account at...,Phishing
3,4,hr@company.com,hr@company.com,hr@company.com,Interview Schedule,Your interview is scheduled for Monday. Please...,Legitimate
4,5,service@bank-secure.net,verify@gmail.com,bounce@bank-secure.net,Bank Account Suspended,Your bank account has been suspended. Verify i...,Phishing


In [5]:
# ================================
# EXPERIMENT 1.4 - PHISHING EMAIL ANALYSIS
# Steps 5-12 (Ready to Paste in Google Colab)
# ================================

import pandas as pd
import re
from urllib.parse import urlparse

# -------------------------------
# Load Dataset
# -------------------------------
df = pd.read_csv("phishing_emails.csv")

# -------------------------------
# Step 5 - Analyze Email Headers
# -------------------------------

trusted_domains = [
    "paypal.com",
    "amazon.com",
    "google.com",
    "company.com",
    "github.com",
    "linkedin.com"
]

def header_analysis(row):
    issues = []

    from_domain = row["From"].split("@")[-1].lower()
    reply_domain = row["Reply_To"].split("@")[-1].lower()
    return_domain = row["Return_Path"].split("@")[-1].lower()

    if from_domain != reply_domain:
        issues.append("Reply-To mismatch")

    if from_domain != return_domain:
        issues.append("Return-Path mismatch")

    return ", ".join(issues) if issues else "No Issues"

df["Header_Analysis"] = df.apply(header_analysis, axis=1)

# -------------------------------
# Step 6 - Verify Sender Address
# -------------------------------

def verify_sender(email):

    domain = email.split("@")[-1].lower()

    if domain in trusted_domains:
        return "Legitimate"

    return "Suspicious"

df["Sender_Status"] = df["From"].apply(verify_sender)

# -------------------------------
# Step 7 - Extract URLs
# -------------------------------

url_pattern = r'https?://[^\s"]+'

def extract_urls(text):

    urls = re.findall(url_pattern, str(text))

    return urls

df["URLs"] = df["Body"].apply(extract_urls)

# -------------------------------
# Extract Domains
# -------------------------------

def extract_domains(urls):

    domains = []

    for url in urls:
        domains.append(urlparse(url).netloc)

    return domains

df["Domains"] = df["URLs"].apply(extract_domains)

# -------------------------------
# Step 8 - Classify URLs
# -------------------------------

def classify_urls(urls):

    result = []

    for url in urls:

        score = 0

        if url.startswith("http://"):
            score += 20

        domain = urlparse(url).netloc.lower()

        if domain not in trusted_domains:
            score += 30

        if any(x in domain for x in ["xyz","secure","verify","login","support"]):
            score += 20

        if len(url) > 50:
            score += 10

        if score >= 40:
            result.append("Suspicious")
        else:
            result.append("Legitimate")

    return result

df["URL_Status"] = df["URLs"].apply(classify_urls)

# -------------------------------
# Step 9 - Detect Spoofed Sender
# -------------------------------

def spoof_check(row):

    from_domain = row["From"].split("@")[-1].lower()
    reply_domain = row["Reply_To"].split("@")[-1].lower()

    if from_domain != reply_domain:
        return "Yes"

    return "No"

df["Spoofed_Sender"] = df.apply(spoof_check, axis=1)

# -------------------------------
# Step 10 - Calculate Risk Score
# -------------------------------

def calculate_score(row):

    score = 0

    if row["Sender_Status"] == "Suspicious":
        score += 25

    if row["Spoofed_Sender"] == "Yes":
        score += 20

    for status in row["URL_Status"]:
        if status == "Suspicious":
            score += 25

    if "Reply-To mismatch" in row["Header_Analysis"]:
        score += 15

    if score >= 60:
        level = "High"

    elif score >= 30:
        level = "Medium"

    else:
        level = "Low"

    return pd.Series([score, level])

df[["Risk_Score","Risk_Level"]] = df.apply(calculate_score, axis=1)

# -------------------------------
# Step 11 - Generate Report
# -------------------------------

report = df[[
    "Email_ID",
    "From",
    "Subject",
    "Header_Analysis",
    "Sender_Status",
    "URLs",
    "Domains",
    "URL_Status",
    "Spoofed_Sender",
    "Risk_Score",
    "Risk_Level",
    "Label"
]]

print("\n===============================")
print("PHISHING EMAIL REPORT")
print("===============================\n")

print(report)

# Save Report
report.to_csv("Phishing_Email_Report.csv", index=False)

# -------------------------------
# Step 12 - Summary Statistics
# -------------------------------

total = len(df)

phishing = (df["Label"]=="Phishing").sum()

legitimate = (df["Label"]=="Legitimate").sum()

spoofed = (df["Spoofed_Sender"]=="Yes").sum()

high = (df["Risk_Level"]=="High").sum()

medium = (df["Risk_Level"]=="Medium").sum()

low = (df["Risk_Level"]=="Low").sum()

suspicious_urls = 0

for row in df["URL_Status"]:
    suspicious_urls += row.count("Suspicious")

print("\n===============================")
print("SUMMARY STATISTICS")
print("===============================")

print("Total Emails            :", total)
print("Phishing Emails         :", phishing)
print("Legitimate Emails       :", legitimate)
print("Spoofed Senders         :", spoofed)
print("Suspicious URLs         :", suspicious_urls)
print("High Risk Emails        :", high)
print("Medium Risk Emails      :", medium)
print("Low Risk Emails         :", low)
print("Average Risk Score      :", round(df["Risk_Score"].mean(),2))

print("\nReport saved as: Phishing_Email_Report.csv")


PHISHING EMAIL REPORT

   Email_ID                              From                     Subject  \
0         1                support@paypa1.com  Verify Your PayPal Account   
1         2               security@amazon.com   Amazon Order Confirmation   
2         3       admin@micr0soft-support.com    Microsoft Security Alert   
3         4                    hr@company.com          Interview Schedule   
4         5           service@bank-secure.net      Bank Account Suspended   
5         6               no-reply@google.com     Google Password Changed   
6         7  support@appleid-verification.com             Apple ID Locked   
7         8             newsletter@github.com              GitHub Updates   
8         9       billing@netflix-support.xyz      Netflix Payment Failed   
9        10               alerts@linkedin.com      New Login Notification   

     Header_Analysis Sender_Status                                  URLs  \
0  Reply-To mismatch    Suspicious     [https://payp